# YgoMaster-PackBuilder — Project Assembly Tests

Test offline del `ProjectDefinition` assembler con fake YGOPRODeck client.

## 1. Repository root

In [1]:
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Impossibile individuare la root della repository.")

print("Repository root:", repo_root)

Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [2]:
from copy import deepcopy

from ygomaster_packbuilder.assembly import assemble_project_definition
from ygomaster_packbuilder.domain import PackType, ProjectDefinition, Rarity
from ygomaster_packbuilder.manifest.models import (
    ManifestPack,
    ManifestPackSource,
    ManifestPlayer,
    ManifestProjectSettings,
    ManifestSharedPack,
    ProjectManifest,
)
from ygomaster_packbuilder.mapping import load_rarity_mapping
from ygomaster_packbuilder.policies.models import (
    PoliciesConfig,
    PolicyDefinition,
    PolicyRarityDistribution,
    PolicyTenPackGuarantee,
    PolicyUltraRarePity,
)

print("Imports OK")

Imports OK


## 3. Fake YGOPRODeck client

In [3]:
class FakeYgoProDeckClient:
    def __init__(self, responses):
        self.responses = responses
        self.calls = []

    def fetch_custom_pack_raw(self, external_id):
        self.calls.append(external_id)
        return deepcopy(self.responses[external_id])


def make_raw_pack(pack_name, card_base_id, *, include_secret=False):
    rarities = ["Common", "Rare", "Super Rare", "Ultra Rare"]
    if include_secret:
        rarities.append("Secret Rare")

    return [
        {
            "username": "test-user",
            "pack_name": pack_name,
            "pack_description": f"{pack_name} description",
            "img_url": str(card_base_id),
            "code": f"TEST-{i}",
            "name": f"{pack_name} Card {i}",
            "type": "Effect Monster",
            "id": card_base_id + i,
            "pack_views": 0,
            "card_rarity": rarity,
        }
        for i, rarity in enumerate(rarities)
    ]

print("Fake client ready")

Fake client ready


## 4. Manifest artificiale

In [4]:
def mp(key, external_id, opening_policy=None):
    return ManifestPack(
        key=key,
        source=ManifestPackSource(
            provider="ygoprodeck",
            external_id=external_id,
        ),
        opening_policy=opening_policy,
    )

player_a = ManifestPlayer(
    key="player-a",
    display_name="Player A",
    main_pack=mp("player-a-main", "main-a"),
    mini_packs=[
        mp("player-a-mini-1", "mini-a-1"),
        mp("player-a-mini-2", "mini-a-2"),
        mp("player-a-mini-3", "mini-a-3", "configuration-b"),
    ],
)

player_b = ManifestPlayer(
    key="player-b",
    display_name="Player B",
    main_pack=mp("player-b-main", "main-b"),
    mini_packs=[
        mp("player-b-mini-1", "mini-b-1"),
        mp("player-b-mini-2", "mini-b-2"),
        mp("player-b-mini-3", "mini-b-3"),
    ],
)

shared = ManifestSharedPack(
    key="shared-staples",
    type="shared_staples",
    source=ManifestPackSource(
        provider="ygoprodeck",
        external_id="shared",
    ),
    opening_policy=None,
)

manifest = ProjectManifest(
    settings=ManifestProjectSettings(
        name="Assembly Test Project",
        default_opening_policy="configuration-a",
    ),
    players=[player_a, player_b],
    shared_packs=[shared],
)

print(manifest)

ProjectManifest(settings=ManifestProjectSettings(name='Assembly Test Project', default_opening_policy='configuration-a'), players=[ManifestPlayer(key='player-a', display_name='Player A', main_pack=ManifestPack(key='player-a-main', source=ManifestPackSource(provider='ygoprodeck', external_id='main-a'), opening_policy=None), mini_packs=[ManifestPack(key='player-a-mini-1', source=ManifestPackSource(provider='ygoprodeck', external_id='mini-a-1'), opening_policy=None), ManifestPack(key='player-a-mini-2', source=ManifestPackSource(provider='ygoprodeck', external_id='mini-a-2'), opening_policy=None), ManifestPack(key='player-a-mini-3', source=ManifestPackSource(provider='ygoprodeck', external_id='mini-a-3'), opening_policy='configuration-b')]), ManifestPlayer(key='player-b', display_name='Player B', main_pack=ManifestPack(key='player-b-main', source=ManifestPackSource(provider='ygoprodeck', external_id='main-b'), opening_policy=None), mini_packs=[ManifestPack(key='player-b-mini-1', source=Man

## 5. Policies artificiali

In [5]:
def dist(n, r, sr, ur):
    return PolicyRarityDistribution(
        normal=n,
        rare=r,
        super_rare=sr,
        ultra_rare=ur,
    )

def make_policy(key):
    return PolicyDefinition(
        key=key,
        normal_slot_distribution=dist(55.0, 35.0, 7.5, 2.5),
        ten_pack_guarantee=PolicyTenPackGuarantee(
            enabled=True,
            final_slot_distribution=dist(0.0, 0.0, 80.0, 20.0),
        ),
        ultra_rare_pity=PolicyUltraRarePity(
            enabled=True,
            guaranteed_slot_distribution=dist(0.0, 0.0, 0.0, 100.0),
        ),
    )

policies = PoliciesConfig(
    policies=[
        make_policy("configuration-a"),
        make_policy("configuration-b"),
    ]
)

print(policies)

PoliciesConfig(policies=[PolicyDefinition(key='configuration-a', normal_slot_distribution=PolicyRarityDistribution(normal=55.0, rare=35.0, super_rare=7.5, ultra_rare=2.5), ten_pack_guarantee=PolicyTenPackGuarantee(enabled=True, final_slot_distribution=PolicyRarityDistribution(normal=0.0, rare=0.0, super_rare=80.0, ultra_rare=20.0)), ultra_rare_pity=PolicyUltraRarePity(enabled=True, guaranteed_slot_distribution=PolicyRarityDistribution(normal=0.0, rare=0.0, super_rare=0.0, ultra_rare=100.0))), PolicyDefinition(key='configuration-b', normal_slot_distribution=PolicyRarityDistribution(normal=55.0, rare=35.0, super_rare=7.5, ultra_rare=2.5), ten_pack_guarantee=PolicyTenPackGuarantee(enabled=True, final_slot_distribution=PolicyRarityDistribution(normal=0.0, rare=0.0, super_rare=80.0, ultra_rare=20.0)), ultra_rare_pity=PolicyUltraRarePity(enabled=True, guaranteed_slot_distribution=PolicyRarityDistribution(normal=0.0, rare=0.0, super_rare=0.0, ultra_rare=100.0)))])


## 6. Source data, rarity mapping e YDK mapping

In [6]:
responses = {
    "main-a": make_raw_pack("Main A", 1000, include_secret=True),
    "mini-a-1": make_raw_pack("Mini A1", 1100, include_secret=True),
    "mini-a-2": make_raw_pack("Mini A2", 1200),
    "mini-a-3": make_raw_pack("Mini A3", 1300),
    "main-b": make_raw_pack("Main B", 2000),
    "mini-b-1": make_raw_pack("Mini B1", 2100),
    "mini-b-2": make_raw_pack("Mini B2", 2200),
    "mini-b-3": make_raw_pack("Mini B3", 2300),
    "shared": make_raw_pack("Shared Staples", 3000, include_secret=True),
}

client = FakeYgoProDeckClient(responses)

rarity_mapping = load_rarity_mapping(
    repo_root / "config" / "rarity_mapping.yaml"
)

all_ydk_ids = {
    int(row["id"])
    for rows in responses.values()
    for row in rows
}

ydk_mapping = {
    ydk_id: ydk_id + 10000
    for ydk_id in all_ydk_ids
}

print("Prepared", len(responses), "fake packs")

Prepared 9 fake packs


## 7. Assemblaggio

In [7]:
project = assemble_project_definition(
    manifest=manifest,
    policies=policies,
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    ygoprodeck_client=client,
)

assert isinstance(project, ProjectDefinition)
print(project)

ProjectDefinition(name='Assembly Test Project', players=[PlayerDefinition(key='player-a', display_name='Player A', main_pack_key='player-a-main', mini_pack_keys=['player-a-mini-1', 'player-a-mini-2', 'player-a-mini-3']), PlayerDefinition(key='player-b', display_name='Player B', main_pack_key='player-b-main', mini_pack_keys=['player-b-mini-1', 'player-b-mini-2', 'player-b-mini-3'])], packs=[PackDefinition(key='player-a-main', name='Main A', description='Main A description', type=<PackType.PERSONAL_MAIN: 'personal_main'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='main-a'), expected_card_count=100, booster_size=7, opening_policy_key='configuration-a', owner_key='player-a', parent_pack_key=None, cards=[Card(name='Main A Card 0', ydk_id=1000, source_rarity='Common', master_duel_id=11000, rarity=<Rarity.N: 'N'>), Card(name='Main A Card 1', ydk_id=1001, source_rarity='Rare', master_duel_id=11001, rarity=<Rarity.R: 'R'>), Card(name='Main A Card 2', y

## 8. Metadata e conteggi

In [8]:
assert project.name == "Assembly Test Project"
assert len(project.players) == 2
assert len(project.packs) == 9
assert len(project.opening_policies) == 2

print("Project metadata OK")

Project metadata OK


## 9. Ordine fetch e pack

In [9]:
expected_calls = [
    "main-a",
    "mini-a-1",
    "mini-a-2",
    "mini-a-3",
    "main-b",
    "mini-b-1",
    "mini-b-2",
    "mini-b-3",
    "shared",
]

expected_pack_keys = [
    "player-a-main",
    "player-a-mini-1",
    "player-a-mini-2",
    "player-a-mini-3",
    "player-b-main",
    "player-b-mini-1",
    "player-b-mini-2",
    "player-b-mini-3",
    "shared-staples",
]

assert client.calls == expected_calls
assert [pack.key for pack in project.packs] == expected_pack_keys

print("Deterministic order OK")

Deterministic order OK


## 10. PlayerDefinition e relazioni

In [10]:
player = project.players[0]

assert player.key == "player-a"
assert player.main_pack_key == "player-a-main"
assert player.mini_pack_keys == [
    "player-a-mini-1",
    "player-a-mini-2",
    "player-a-mini-3",
]

packs = {pack.key: pack for pack in project.packs}

assert packs["player-a-main"].type is PackType.PERSONAL_MAIN
assert packs["player-a-main"].owner_key == "player-a"
assert packs["player-a-main"].parent_pack_key is None

for key in player.mini_pack_keys:
    assert packs[key].type is PackType.PERSONAL_MINI
    assert packs[key].owner_key == "player-a"
    assert packs[key].parent_pack_key == "player-a-main"

assert packs["shared-staples"].type is PackType.SHARED_STAPLES
assert packs["shared-staples"].owner_key is None
assert packs["shared-staples"].parent_pack_key is None

print("Relationships OK")

Relationships OK


## 11. Opening policy inheritance e override

In [11]:
assert packs["player-a-main"].opening_policy_key == "configuration-a"
assert packs["player-a-mini-1"].opening_policy_key == "configuration-a"
assert packs["player-a-mini-3"].opening_policy_key == "configuration-b"
assert packs["shared-staples"].opening_policy_key == "configuration-a"

print("Opening-policy resolution OK")

Opening-policy resolution OK


## 12. Policies convertite nel domain

In [12]:
assert [p.key for p in project.opening_policies] == [
    "configuration-a",
    "configuration-b",
]

config_a = project.opening_policies[0]
assert config_a.normal_slot_distribution.normal == 55.0
assert config_a.ten_pack_guarantee.final_slot_distribution.super_rare == 80.0
assert config_a.ultra_rare_pity.guaranteed_slot_distribution.ultra_rare == 100.0

print("Policy conversion OK")

Policy conversion OK


## 13. Secret Rare propagata correttamente

In [13]:
main_secret = next(
    c for c in packs["player-a-main"].cards
    if c.source_rarity == "Secret Rare"
)
mini_secret = next(
    c for c in packs["player-a-mini-1"].cards
    if c.source_rarity == "Secret Rare"
)
shared_secret = next(
    c for c in packs["shared-staples"].cards
    if c.source_rarity == "Secret Rare"
)

assert main_secret.rarity is Rarity.N
assert mini_secret.rarity is Rarity.N
assert shared_secret.rarity is Rarity.UR

print("Secret Rare overrides OK")

Secret Rare overrides OK


## 14. Master Duel IDs

In [14]:
for pack in project.packs:
    for card in pack.cards:
        assert card.master_duel_id == card.ydk_id + 10000

print("Master Duel IDs OK")

Master Duel IDs OK


## 15. L'assembler non valida il numero di carte

In [15]:
assert len(packs["player-a-main"].cards) != (
    packs["player-a-main"].expected_card_count
)
assert packs["player-a-main"].expected_card_count == 100

print(
    "Actual:",
    len(packs["player-a-main"].cards),
    "Expected:",
    packs["player-a-main"].expected_card_count,
)

Actual: 5 Expected: 100


## 16. Provider non supportato

In [16]:
bad_manifest = deepcopy(manifest)
bad_manifest.players[0].main_pack.source.provider = "unsupported-provider"

bad_client = FakeYgoProDeckClient(responses)

try:
    assemble_project_definition(
        manifest=bad_manifest,
        policies=policies,
        rarity_mapping=rarity_mapping,
        ydk_mapping=ydk_mapping,
        ygoprodeck_client=bad_client,
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Provider non supportato non rifiutato.")

assert bad_client.calls == []

Expected ValueError: Unsupported source provider: 'unsupported-provider'


## 17. Shared pack type non supportato

In [17]:
bad_manifest = deepcopy(manifest)
bad_manifest.shared_packs[0].type = "unsupported_shared_type"

bad_client = FakeYgoProDeckClient(responses)

try:
    assemble_project_definition(
        manifest=bad_manifest,
        policies=policies,
        rarity_mapping=rarity_mapping,
        ydk_mapping=ydk_mapping,
        ygoprodeck_client=bad_client,
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Shared pack type non supportato non rifiutato.")

assert bad_client.calls[-1] == "shared"
print("Unsupported shared type rejected")

Expected ValueError: Unsupported shared pack type: 'unsupported_shared_type'.
Unsupported shared type rejected
